In [21]:
# --- MÓDULO DE INGESTÃO: ARRECADAÇÃO MUNICIPAL (SICONFI / TESOURO NACIONAL) ---

import pandas as pd

# Defino o caminho para os dados históricos de arrecadação municipal.
# Este dataset é fundamental para o cálculo do indicador de dependência fiscal.
caminho_do_arquivo = r'C:\Users\fabio\TCC\2_Arrecadação Municipal\arrecadacao-2005-a-2017.xlsx'

# REALIZO O PARSING DA PLANILHA EXCEL:
# skiprows=5: Ignora o cabeçalho institucional/metadados da planilha original.
# skipfooter=1: Remove notas de rodapé que poderiam gerar registros nulos (NaN).
df = pd.read_excel(
    caminho_do_arquivo,
    skiprows=5,
    skipfooter=1
)

# DIAGNÓSTICO INICIAL:
# Verificação da estrutura de colunas e integridade das primeiras observações.
print("--- Visualização dos Registros Iniciais (Arrecadação) ---")
print(df.head())

print("\n--- Metadados do Dataset ---")
df.info()

            MUNICIPIOS  UF        2005        2006        2007        2008  \
0      ABADIA DE GOIAS  GO   748293.01   743933.43   947889.34  1326710.17   
1  ABADIA DOS DOURADOS  MG   377868.56   367432.40   448066.69   499392.49   
2            ABADIANIA  GO   533029.95   685872.14   863835.95  1230150.65   
3               ABAETE  MG  4179358.65  5198049.55  5868576.82  5440673.41   
4           ABAETETUBA  PA  2081937.34  2375672.21  2885653.45  3740444.69   

         2009        2010        2011         2012         2013         2014  \
0  1481737.43  2380864.09  3643560.50   4534735.71   5877249.54   7841338.72   
1   556702.73   683420.91  1207955.63   2743175.80   3267699.57   3291358.53   
2  1368021.86  1995984.56  2996679.83   6206806.65   7117866.66   7868537.40   
3  5523713.19  6526988.68  8005829.32  11455173.65  13140499.09  14624455.60   
4  4868675.36  5224109.14  6967024.70  15160086.13  17680084.77  20342911.70   

          2015         2016         2017  
0   730

In [22]:
# --- MAPEAMENTO E INVENTÁRIO DE ATRIBUTOS (COLUNAS DE RECEITA) ---

# Realizo a inspeção completa do índice de colunas do dataset. 
# Este procedimento é vital para identificar as variáveis de interesse (Receitas Correntes, 
# Tributárias e Transferências) e planejar a renomeação para o padrão snake_case.

print("--- Inventário de Colunas: Dados Brutos de Arrecadação ---")

# Exibo a listagem completa para auditoria de nomes e identificação de 
# possíveis caracteres especiais ou quebras de linha vindas do Excel.
print(df.columns.tolist())

# Diagnóstico de Dimensão:
# Verifico a largura do dataset para entender a granularidade das contas contábeis disponíveis.
print(f"\nTotal de colunas identificadas: {len(df.columns)}")

Index(['MUNICIPIOS', 'UF', '2005', '2006', '2007', '2008', '2009', '2010',
       '2011', '2012', '2013', '2014', '2015', '2016', '2017'],
      dtype='object')


In [23]:
# --- REESTRUTURAÇÃO DE DADOS (RESHAPING: WIDE TO LONG) ---

# Para viabilizar a análise de Séries Temporais e Painéis de Dados (Panel Data), 
# realizo o "Melt" do DataFrame. Este procedimento converte a estrutura horizontal 
# de anos em uma estrutura vertical, essencial para a integração com o Stata.

print("Iniciando a transformação da estrutura (Wide para Long)...")

# Utilizo a função .melt() para normalizar a base:
# id_vars: Dimensões fixas que identificam a unidade de observação (Município/UF).
# var_name: Consolida as colunas de anos em uma única variável temporal ('Ano').
# value_name: Atribui os montantes financeiros à variável métrica ('Arrecadacao').
df_longo = df.melt(
    id_vars=['MUNICIPIOS', 'UF'],
    var_name='Ano',
    value_name='Arrecadacao'
)

# DIAGNÓSTICO DE TRANSFORMAÇÃO:
# Verifico se a volumetria de linhas aumentou proporcionalmente ao número de anos,
# confirmando a integridade do processo de reshaping.
print(f"--- Painel Verticalizado Concluído ---")
print(f"Dimensão Final: {df_longo.shape[0]} linhas por {df_longo.shape[1]} colunas.")
print(df_longo.head())

In [24]:
print(df_longo.head())
print(df_longo.tail())

            MUNICIPIOS  UF   Ano  Arrecadacao
0      ABADIA DE GOIAS  GO  2005    748293.01
1  ABADIA DOS DOURADOS  MG  2005    377868.56
2            ABADIANIA  GO  2005    533029.95
3               ABAETE  MG  2005   4179358.65
4           ABAETETUBA  PA  2005   2081937.34
        MUNICIPIOS  UF   Ano  Arrecadacao
72418  XIQUE-XIQUE  BA  2017   7491900.40
72419       ZABELE  PB  2017     77315.51
72420     ZACARIAS  SP  2017   1670122.17
72421      ZE DOCA  MA  2017   5270496.12
72422       ZORTEA  SC  2017   1220668.80


In [25]:
# --- MÓDULO DE AUTOMAÇÃO: CONSOLIDAÇÃO HISTÓRICA MULTI-ARQUIVO (2018-2024) ---

# Implemento um pipeline automatizado para processar os arquivos anuais de arrecadação.
# O foco é a normalização de chaves e a padronização de nomenclatura (Dicionário Mestre),
# garantindo a integridade do painel de dados durante a concatenação.

# 1. DICIONÁRIO MESTRE DE MAPEAMENTO (RENAMING DICTIONARY)
# Padronizo as variações ortográficas e de capitalização encontradas nos relatórios
# do Siconfi, assegurando que variáveis idênticas possuam o mesmo identificador.
master_rename_dict = {
    'MUNICÍPIOS': 'MUNICIPIOS',   
    'Município': 'MUNICIPIOS',    
    'ARRECADAÇÃO': 'Arrecadacao', 
    'Arrecadação': 'Arrecadacao'  
}

# 2. INICIALIZAÇÃO DA ESTRUTURA DE ARMAZENAMENTO
# Utilizo uma lista para otimizar a performance da memória antes da concatenação final.
lista_outros_anos = []

# 3. CONFIGURAÇÃO DE DIRETÓRIOS
pasta_dados = r'C:\Users\fabio\TCC\2_ArrecadaÇÃO Municipal'

print("--- INICIANDO PROCESSAMENTO EM LOTE (BATCH PROCESSING) ---")

# 4. PIPELINE DE REPETIÇÃO (ROBÔ DE TRATAMENTO)
for ano in range(2018, 2025):
    
    print(f'Processando horizonte temporal: {ano}')
    
    nome_arquivo = f'arrecadacao-{ano}.xlsx'
    caminho_completo = f'{pasta_dados}\\{nome_arquivo}'

    try:
        # Ingestão de dados com tratamento de metadados (skiprows/footer)
        df_ano_bruto = pd.read_excel(
            caminho_completo,
            skiprows=5,
            skipfooter=1
        )
        
        # Injeção da variável temporal para controle de painel
        df_ano_bruto['Ano'] = ano
        
        # 5. NORMALIZAÇÃO DE ATRIBUTOS
        # Aplico o mapeamento mestre para garantir que todos os DataFrames possuam
        # o mesmo esquema (schema) de colunas antes do merge.
        df_ano_corrigido = df_ano_bruto.rename(columns=master_rename_dict)
        
        # Armazenamento temporário do DataFrame tratado
        lista_outros_anos.append(df_ano_corrigido)
        
        print(f'   -> Status: Processamento Concluído com Sucesso.')
        
    except FileNotFoundError:
        print(f'   -> [AVISO]: Arquivo do ano {ano} não localizado. Ignorando entrada.')
    except Exception as e:
        print(f'   -> [ERRO]: Falha crítica no processamento do ano {ano}: {e}')

print("--- PIPELINE DE CONSOLIDAÇÃO CONCLUÍDO ---")

--- INICIANDO O LOOP CORRIGIDO (2018-2024) ---
Processando ano: 2018


C:\Users\fabio\anaconda3\Lib\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


   -> Sucesso!
Processando ano: 2019
   -> Sucesso!
Processando ano: 2020
   -> Sucesso!
Processando ano: 2021
   -> Sucesso!
Processando ano: 2022
   -> Sucesso!
Processando ano: 2023
   -> Sucesso!
Processando ano: 2024
   -> Sucesso!
--- LOOP CONCLUÍDO ---


C:\Users\fabio\anaconda3\Lib\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


In [26]:
# --- CONSOLIDAÇÃO DO PAINEL HISTÓRICO (CONCATENAÇÃO 2018-2024) ---

# Após o processamento individual em lote, realizo a união vertical (stacking) 
# dos DataFrames. Este passo consolida o horizonte temporal recente do TCC,
# preparando os dados para a integração com a série histórica anterior.

if lista_outros_anos:
    # Executo a concatenação eficiente da lista de objetos DataFrame.
    # A variável 'df_outros_anos' agora representa o painel consolidado do período.
    df_outros_anos = pd.concat(lista_outros_anos)

    print("--- Relatório de Consolidação: Período 2018-2024 ---")
    
    # Verificação de Integridade de Atributos:
    # Garanto que a normalização via dicionário mestre funcionou e não gerou 
    # colunas duplicadas ou fragmentadas (ex: 'Município' vs 'MUNICIPIOS').
    print("\nVerificação de Schema (Colunas Unificadas):")
    print(df_outros_anos.columns.tolist()) 
    
    # Auditoria de Continuidade Temporal:
    # Valido os extremos da série para assegurar que o empilhamento respeitou a cronologia.
    print("\nAuditoria de Início de Série (Expectativa: 2018):")
    print(df_outros_anos.head())
    
    print("\nAuditoria de Fim de Série (Expectativa: 2024):")
    print(df_outros_anos.tail())
else:
    print("\n[FALHA DE PROCESSO]: Nenhuma estrutura de dados foi populada durante o loop.")

--- Dados de 2018-2024 Consolidados (Corretamente) ---
Colunas devem ser únicas:
['MUNICIPIOS', 'UF', 'Arrecadacao', 'Ano']

Início (deve ser 2018):
            MUNICIPIOS  UF  Arrecadacao   Ano
0      ABADIA DE GOIAS  GO   9509713.49  2018
1  ABADIA DOS DOURADOS  MG   3457362.00  2018
2            ABADIANIA  GO   8996131.56  2018
3               ABAETE  MG  16060311.20  2018
4           ABAETETUBA  PA  28281622.15  2018

Fim (deve ser 2024):
       MUNICIPIOS  UF  Arrecadacao   Ano
5565  XIQUE-XIQUE  BA   3450774.34  2024
5566       ZABELE  PB     42672.55  2024
5567     ZACARIAS  SP    130748.23  2024
5568      ZE DOCA  MA   2971275.25  2024
5569       ZORTEA  SC    105714.25  2024


In [27]:
# --- CONSOLIDAÇÃO FINAL DA SÉRIE HISTÓRICA DE ARRECADAÇÃO (2005-2024) ---

# Nesta etapa final, realizo a integração das duas janelas temporais processadas: 
# o painel histórico (2005-2017) e o painel recente (2018-2024). 
# O resultado é um dataset unificado e pronto para a análise de impacto de longo prazo.

# 1. PREPARAÇÃO DOS COMPONENTES DE SÉRIE
# Reúno os objetos DataFrame tratados em uma estrutura de lista para concatenação.
tabelas_finais = [df_longo, df_outros_anos]

# 2. UNIÃO VERTICAL (CONCATENAÇÃO INTEGRADA)
# Executo o empilhamento das séries, garantindo a continuidade do fluxo de dados 
# entre as diferentes gestões orçamentárias registradas no Siconfi.
df_arrecadacao_final = pd.concat(tabelas_finais)

# --- PROTOCOLO DE VERIFICAÇÃO E AUDITORIA FINAL ---

print("--- DIAGNÓSTICO DE CONSOLIDAÇÃO: SÉRIE 2005-2024 ---")

# Validação de Dimensão:
# Confirmo a volumetria total para assegurar que não houve perda de registros 
# durante a transição dos períodos.
print(f'Matriz Final (Observações x Variáveis): {df_arrecadacao_final.shape}')

# Verificação de Cronologia Inicial:
# Certifico que a série inicia-se em 2005, capturando o cenário pré-tratamento.
print("\nAuditoria de Cronologia Inicial (Expectativa: 2005):")
print(df_arrecadacao_final.head())

# Verificação de Cronologia Final:
# Certifico que a série estende-se até 2024, cobrindo o período mais recente disponível.
print("\nAuditoria de Cronologia Final (Expectativa: 2024):")
print(df_arrecadacao_final.tail())

--- TABELA DE ARRECADAÇÃO CONSOLIDADA (2005-2024) ---
Formato final da tabela: (111383, 4)

Início (deve ser 2005):
            MUNICIPIOS  UF   Ano  Arrecadacao
0      ABADIA DE GOIAS  GO  2005    748293.01
1  ABADIA DOS DOURADOS  MG  2005    377868.56
2            ABADIANIA  GO  2005    533029.95
3               ABAETE  MG  2005   4179358.65
4           ABAETETUBA  PA  2005   2081937.34

Fim (deve ser 2024):
       MUNICIPIOS  UF   Ano  Arrecadacao
5565  XIQUE-XIQUE  BA  2024   3450774.34
5566       ZABELE  PB  2024     42672.55
5567     ZACARIAS  SP  2024    130748.23
5568      ZE DOCA  MA  2024   2971275.25
5569       ZORTEA  SC  2024    105714.25


In [29]:
# --- PERSISTÊNCIA DOS DADOS E GERAÇÃO DO DATASET FINAL (OUTPUT) ---

# Concluo o pipeline de Arrecadação Municipal exportando a série histórica 
# consolidada (2005-2024). Este arquivo servirá como a fonte primária para 
# o cálculo de indicadores de dependência fiscal e capacidade arrecadatória.

# 1. DEFINIÇÃO DO REPOSITÓRIO DE SAÍDA
# Armazeno o dataset na pasta raiz do projeto para facilitar a integração 
# com os módulos subsequentes de análise econométrica.
caminho_saida_csv = r'C:\Users\fabio\TCC\arrecadacao_CONSOLIDADA_FINAL.csv'

# 2. EXPORTAÇÃO DO DATASET (SERIALIZAÇÃO)
# Utilizo o parâmetro index=False para evitar a inclusão de metadados redundantes 
# do Pandas, mantendo a estrutura limpa para leitura direta no Stata ou R.
df_arrecadacao_final.to_csv(
    caminho_saida_csv,
    index=False
)

print(f"--- PROTOCOLO DE EXPORTAÇÃO CONCLUÍDO ---")
print(f"Dataset Histórico Consolidado: {caminho_saida_csv}")

--- SUCESSO! ---
Seu arquivo consolidado foi salvo em:
C:\Users\fabio\TCC\arrecadacao_CONSOLIDADA_FINAL.csv
